# Figure notebook 09 — validation, robustness and baselines

One appendix figure per model, built from the bundles that `scripts/build_validation_bundles.py`
re-encodes out of the notebook-09 cluster results — no model, no dataset, no BFT run.

Every figure has the same nine panels, so the five can be read side by side as a table:

| row | claim | panels |
|---|---|---|
| 1 | **is it faithful?** | (a) causal reconstruction against a random-rank floor, an activation-only NMF at the same rank, a rank-matched SVD and exact injection · (b) reconstruction of every node, by depth, with the held-out/in-sample split · (c) NNLS projection round-trip |
| 2 | **is it robust?** | (d) factor stability across NMF seeds · (e) sensitivity to the rank (K*±1) · (f) how much arbor each rank explains |
| 3 | **is it better than the controls?** | (g) fingerprint vs. the network's own activations, dimension-matched, random-projected and against a shuffled-label null · (h) the weight term (arbor NMF) vs. activation-only NMF · (i) vs. pixel attribution |

A panel a model cannot supply is drawn as a dashed empty frame that says why — layer-dict
traces (ViT, SqueezeNet) have no model to re-run, so they have no causal reconstruction.
Model-seed error bars appear on the two headline numbers, (b) and (g), where five models
were retrained.

- data: `figures/figdata/nb09_<exp>_validation.{npz,json}` (from `logs/results/nb09_<exp>.json`)
- render function: `src/paper_figures.py` → `fig_validation`, one entry per model in `FIGURES`
- style/venue: `figstyle.py`, config in `.figstyle/`, captions in `.figstyle/figures.md`
- rebuild the bundles: `python scripts/build_validation_bundles.py`
- redraw every figure of the paper: `python scripts/render_figures.py`

## §0 — Setup

In [ ]:
#%matplotlib inline
import sys, os
sys.path.insert(0, '..')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import figstyle
from src import figdata
from src.paper_figures import FIGURES

plt.rcParams.update({'figure.dpi': 110})
print('bundles available:', ', '.join(figdata.available()))

## §1 — Load the exported validation data

One bundle per experiment. Each is the full notebook-09 result dict: capability flags,
the run config, every measured section (S1–S11, FU1, FU2) and the selected hyperparameters.

In [ ]:
VAL = {name: figdata.load(bundle)
       for name, (bundle, _, _) in FIGURES.items() if bundle.startswith('nb09_')}

for name, D in VAL.items():
    caps = ', '.join(k for k, v in D['caps'].items() if v)
    print(f"{name:32s} {D['label']:14s} n={D['config']['n_samples']:>5d}  caps: {caps}")

## §2 — What is in a bundle

`summary()` prints every leaf with shape and dtype — enough to design a panel without
opening notebook 09.

In [ ]:
figdata.summary('nb09_mlp_even_odd_validation')

## §3 — Figures

One cell per model. All five call the same `fig_validation` render function; the bundle
carries the model name, its capability flags and every number the panels draw.

### figI_validation_mlp_even_odd — MLP 784→8→4→2, MNIST even/odd

The only setting where all nine panels are populated: every node reconstructs causally, the round-trip is near-exact, and the fingerprint beats the activations on the fine-grained (digit) label.

In [ ]:
# figI_validation_mlp_even_odd (appendix) — validation, robustness and baselines
# data: VAL['figI_validation_mlp_even_odd']   (nb09_mlp_even_odd_validation)
bundle, render, mode = FIGURES['figI_validation_mlp_even_odd']
fig = render(VAL['figI_validation_mlp_even_odd'])
figstyle.save_fig(fig, 'figI_validation_mlp_even_odd')

### figJ_validation_mlp_digit — MLP 784→40→20→10, MNIST digits

The hardest faithfulness case: BFT reconstructs well below the rank-matched SVD ceiling (0.65 vs 0.90) and stability sits at the 0.85 gate. Shown as it is.

In [ ]:
# figJ_validation_mlp_digit (appendix) — validation, robustness and baselines
# data: VAL['figJ_validation_mlp_digit']   (nb09_mlp_digit_validation)
bundle, render, mode = FIGURES['figJ_validation_mlp_digit']
fig = render(VAL['figJ_validation_mlp_digit'])
figstyle.save_fig(fig, 'figJ_validation_mlp_digit')

### figK_validation_cnn_cifar — SmallCNN, CIFAR-10

The first conv setting with a causal reconstruction (0.938 at the classifier). Only one node is reconstructable — spatial pooling is not invertible.

In [ ]:
# figK_validation_cnn_cifar (appendix) — validation, robustness and baselines
# data: VAL['figK_validation_cnn_cifar']   (nb09_cnn_cifar_validation)
bundle, render, mode = FIGURES['figK_validation_cnn_cifar']
fig = render(VAL['figK_validation_cnn_cifar'])
figstyle.save_fig(fig, 'figK_validation_cnn_cifar')

### figL_validation_vit — TinyViT d=32, MNIST even/odd

Layer-dict mode: no model to re-run, so (a) and (c) are empty and (b) comes from the FU2 fc-node measurement — where one FFN node reconstructs catastrophically.

In [ ]:
# figL_validation_vit (appendix) — validation, robustness and baselines
# data: VAL['figL_validation_vit']   (nb09_vit_mnist_validation)
bundle, render, mode = FIGURES['figL_validation_vit']
fig = render(VAL['figL_validation_vit'])
figstyle.save_fig(fig, 'figL_validation_vit')

### figM_validation_imagenet — SqueezeNet 1.1 spine, ImageNet (8 categories)

The flagship separability result: at identical dimensionality the fingerprint gets 4x the silhouette of the activations, and the random projection rules out "PCA merely damaged the activations".

In [ ]:
# figM_validation_imagenet (appendix) — validation, robustness and baselines
# data: VAL['figM_validation_imagenet']   (nb09_imagenet_cnn_validation)
bundle, render, mode = FIGURES['figM_validation_imagenet']
fig = render(VAL['figM_validation_imagenet'])
figstyle.save_fig(fig, 'figM_validation_imagenet')

## §4 — Cross-model summary

The numbers the five figures are built from, as one table — this is what the appendix text
quotes, and what makes the honest negatives easy to check.

In [ ]:
rows = []
for name, D in VAL.items():
    sep, a1 = D['separability']['by_fine'], D['A1_weight_vs_activation']
    rec = D['recon'] if isinstance(D['recon'], dict) else None
    stab = [v['mean'] for v in D['stability']['per_layer'].values()]
    rows.append(dict(
        model=D['label'],
        n=int(D['config']['n_samples']),
        causal_r2=f"{rec['overall']['preact_r2']['median']:.3f}" if rec else '--',
        roundtrip=(f"{D['roundtrip']['median']:.3f}"
                   if isinstance(D['roundtrip'], dict) else '--'),
        min_stab=f'{min(stab):.3f}',
        sil_bft=f"{sep['bft_fingerprint']['silhouette']:.3f}",
        sil_act=f"{sep['act_matched']['silhouette']:.3f}",
        sil_arbor=f"{a1['fingerprint_separability']['arbor_nmf']['silhouette']:.3f}",
        sil_actnmf=f"{a1['fingerprint_separability']['activation_nmf']['silhouette']:.3f}"))

hdr = list(rows[0])
w = {k: max(len(k), *(len(str(r[k])) for r in rows)) for k in hdr}
print('  '.join(k.ljust(w[k]) for k in hdr))
for r in rows:
    print('  '.join(str(r[k]).ljust(w[k]) for k in hdr))